# End-to-End Lakehouse Pipeline (PoC)

This notebook demonstrates the Medallion Architecture flow based on our `senior-data-engineer` and `spark-optimization` principles:
1. **Bronze (Raw)**: Extract **schedule**, **matchsheet** (team stats), and **lineup** (player stats) via ESPN soccerdata and save raw JSON directly to MinIO.
2. **Silver (Analytics)**: Use PySpark to read the Bronze JSONs, derive scores from lineup goals, join all data, and save as idempotent Apache Iceberg tables.

### 0. Install Dependencies

In [ ]:
# Install Boto3 for MinIO interaction and soccerdata for the ESPN extraction
!pip install -q boto3 soccerdata pandas

### 1. Extract Data (Simulating Airflow Task -> Bronze)

We use all three ESPN methods from soccerdata:
- `read_schedule()` -- match list with home/away teams and dates
- `read_matchsheet()` -- team-level stats (possession, shots, cards, passes, tackles, etc.)
- `read_lineup()` -- player-level stats (goals, assists, shots, fouls, position, etc.)

In [ ]:
import os
import json
import time
import boto3
import soccerdata as sd
from soccerdata import _config

LEAGUE = "BRA-Brasileirao"
SEASON = 2024

# Ensure the league mapping exists for ESPN
if LEAGUE not in _config.LEAGUE_DICT:
    _config.LEAGUE_DICT[LEAGUE] = {"ESPN": "bra.1"}

# Initialize ESPN reader
print(f"Fetching data for {LEAGUE} {SEASON}...")
reader = sd.ESPN(leagues=LEAGUE, seasons=SEASON)

# --- 1a. Schedule (match list) ---
print("Fetching schedule...")
schedule = reader.read_schedule().reset_index()
print(f"  Schedule: {len(schedule)} matches")
print(f"  Columns: {list(schedule.columns)}")

# --- 1b. Matchsheet (team stats per match) ---
print("\nFetching matchsheet (team stats)... this may take a few minutes.")
matchsheet = reader.read_matchsheet().reset_index()
# Drop the roster column (raw JSON blobs, too large for storage)
matchsheet = matchsheet.drop(columns=["roster"], errors="ignore")
print(f"  Matchsheet: {len(matchsheet)} rows (2 per match: home + away)")
print(f"  Columns: {list(matchsheet.columns)}")

# --- 1c. Lineup (player stats per match) ---
print("\nFetching lineup (player stats)... this may take a few minutes.")
lineup = reader.read_lineup().reset_index()
print(f"  Lineup: {len(lineup)} player entries")
print(f"  Columns: {list(lineup.columns)}")

# Convert all DataFrames to JSON strings
schedule_json_str = json.dumps(
    json.loads(schedule.to_json(orient="records", date_format="iso")),
    indent=2, ensure_ascii=False
)
matchsheet_json_str = json.dumps(
    json.loads(matchsheet.to_json(orient="records", date_format="iso")),
    indent=2, ensure_ascii=False
)
lineup_json_str = json.dumps(
    json.loads(lineup.to_json(orient="records", date_format="iso")),
    indent=2, ensure_ascii=False
)

print(f"\nExtraction complete. Ready to upload to Bronze.")

### 2. Upload to MinIO (Bronze Layer)

Three separate raw JSON files: schedule, matchsheet, and lineup.

In [ ]:
# Connect to MinIO internal network via Boto3
MINIO_ENDPOINT = "http://minio:9000" 
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin123"
RAW_BUCKET = "datalake-raw"

s3_client = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    region_name="us-east-1"
)

# Upload all three datasets to Bronze
bronze_base = f"espn/brasileirao/{SEASON}"

# Schedule (match list with teams and dates)
s3_client.put_object(
    Bucket=RAW_BUCKET,
    Key=f"{bronze_base}/schedule.json",
    Body=schedule_json_str
)
print(f"Uploaded schedule to s3://{RAW_BUCKET}/{bronze_base}/schedule.json")

# Matchsheet (team stats: possession, shots, cards, passes, tackles)
s3_client.put_object(
    Bucket=RAW_BUCKET,
    Key=f"{bronze_base}/matchsheet.json",
    Body=matchsheet_json_str
)
print(f"Uploaded matchsheet to s3://{RAW_BUCKET}/{bronze_base}/matchsheet.json")

# Lineup (player stats: goals, assists, shots, fouls, position)
s3_client.put_object(
    Bucket=RAW_BUCKET,
    Key=f"{bronze_base}/lineup.json",
    Body=lineup_json_str
)
print(f"Uploaded lineup to s3://{RAW_BUCKET}/{bronze_base}/lineup.json")

### 3. Spark Initialization & Optimization

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# The Spark session is already configured via spark-defaults.conf to use Iceberg and connect to MinIO.
# Applying Adaptive Query Execution (AQE) best practices for optimizations.
spark = (SparkSession.builder
    .appName("BrasileiraoLakehouseAnalytics")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.skewJoin.enabled", "true")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session Created with AQE enabled.")

### 4. Build Enriched Match Statistics (Bronze -> Silver)

Joins schedule + matchsheet + derived scores from lineup into a single wide `match_statistics` table.

In [ ]:
# Read the three Bronze JSONs from MinIO using S3A
bronze_uri = f"s3a://{RAW_BUCKET}/{bronze_base}"

df_schedule = spark.read.option("multiline", "true").json(f"{bronze_uri}/schedule.json")
df_matchsheet = spark.read.option("multiline", "true").json(f"{bronze_uri}/matchsheet.json")
df_lineup = spark.read.option("multiline", "true").json(f"{bronze_uri}/lineup.json")

print(f"Schedule:   {df_schedule.count()} rows")
print(f"Matchsheet: {df_matchsheet.count()} rows")
print(f"Lineup:     {df_lineup.count()} rows")

# --- Step 1: Derive scores from lineup (sum total_goals per game + team) ---
df_scores = (
    df_lineup
    # Cast total_goals to integer (may be null or string)
    .withColumn("total_goals", F.coalesce(F.col("total_goals").cast("int"), F.lit(0)))
    # Aggregate: sum goals per game + is_home
    .groupBy("game", "is_home")
    .agg(F.sum("total_goals").alias("team_goals"))
)

# Pivot to get home_score and away_score columns per game
df_home_scores = df_scores.filter(F.col("is_home") == True).select(
    F.col("game"), F.col("team_goals").alias("home_score")
)
df_away_scores = df_scores.filter(F.col("is_home") == False).select(
    F.col("game"), F.col("team_goals").alias("away_score")
)

# --- Step 2: Pivot matchsheet to get home/away stats side by side ---
# Stat columns from matchsheet (exclude metadata)
meta_cols = {"league", "season", "game", "team", "is_home", "venue", "attendance", "capacity"}
stat_cols = [c for c in df_matchsheet.columns if c not in meta_cols]

# Home team stats (prefix with home_)
df_home_stats = df_matchsheet.filter(F.col("is_home") == True)
for col_name in stat_cols:
    df_home_stats = df_home_stats.withColumnRenamed(col_name, f"home_{col_name}")
df_home_stats = df_home_stats.select(
    "game", "venue", "attendance", "capacity",
    *[f"home_{c}" for c in stat_cols]
)

# Away team stats (prefix with away_)
df_away_stats = df_matchsheet.filter(F.col("is_home") == False)
for col_name in stat_cols:
    df_away_stats = df_away_stats.withColumnRenamed(col_name, f"away_{col_name}")
df_away_stats = df_away_stats.select(
    "game",
    *[f"away_{c}" for c in stat_cols]
)

# --- Step 3: Join everything into one enriched match table ---
df_silver = (
    df_schedule
    # Join home scores
    .join(df_home_scores, on="game", how="left")
    # Join away scores
    .join(df_away_scores, on="game", how="left")
    # Join home team stats + venue/attendance
    .join(df_home_stats, on="game", how="left")
    # Join away team stats
    .join(df_away_stats, on="game", how="left")
)

# --- Step 4: Add computed analytics columns ---
df_silver = (
    df_silver
    .withColumn("home_score", F.col("home_score").cast("int"))
    .withColumn("away_score", F.col("away_score").cast("int"))
    .withColumn("total_goals", F.col("home_score") + F.col("away_score"))
    .withColumn("goal_diff", F.abs(F.col("home_score") - F.col("away_score")))
    .withColumn("is_draw", F.col("home_score") == F.col("away_score"))
    .withColumn("winner", F.when(
        F.col("home_score") > F.col("away_score"), F.col("home_team")
    ).when(
        F.col("away_score") > F.col("home_score"), F.col("away_team")
    ).otherwise(F.lit("Draw")))
    .withColumn("processed_at", F.current_timestamp())
)

print(f"\nEnriched Silver table: {df_silver.count()} rows, {len(df_silver.columns)} columns")
print(f"Columns: {df_silver.columns}")

# Show a sample of key columns
df_silver.select(
    "game_id", "date", "home_team", "away_team",
    "home_score", "away_score", "total_goals", "winner",
    "venue", "attendance"
).show(5, truncate=False)

### 4b. Build Player Match Statistics (Bronze -> Silver)

Creates a `player_match_stats` Silver table from the lineup data with per-player goals, assists, and performance metrics.

In [ ]:
# Cast numeric columns in the lineup data
numeric_cols = [
    "total_goals", "goal_assists", "shots_on_target", "total_shots",
    "fouls_committed", "fouls_suffered", "yellow_cards", "red_cards",
    "own_goals", "offsides", "appearances", "saves", "shots_faced",
    "goals_conceded"
]

df_player_stats = df_lineup
for col_name in numeric_cols:
    if col_name in df_player_stats.columns:
        # Cast to int, filling nulls with 0
        df_player_stats = df_player_stats.withColumn(
            col_name,
            F.coalesce(F.col(col_name).cast("int"), F.lit(0))
        )

# Add processing timestamp
df_player_stats = df_player_stats.withColumn("processed_at", F.current_timestamp())

print(f"Player Match Stats Silver table: {df_player_stats.count()} rows")
print(f"Columns: {df_player_stats.columns}")

# Show top scorers
print("\nTop scorers across all matches:")
df_player_stats.filter(F.col("total_goals") > 0).select(
    "player", "team", "game", "total_goals", "goal_assists"
).orderBy(F.col("total_goals").desc()).show(10, truncate=False)

### 5. Write to Iceberg (Silver/Warehouse Layer)

Two Silver Iceberg tables:
- `match_statistics` -- enriched match results with team stats
- `player_match_stats` -- per-player per-match performance

In [ ]:
# Create namespace if not exists
spark.sql("CREATE NAMESPACE IF NOT EXISTS lake.analytics")

# --- Write enriched match statistics ---
match_table = "lake.analytics.match_statistics"
df_silver.writeTo(match_table).createOrReplace()
print(f"Successfully wrote {df_silver.count()} rows to Iceberg table {match_table}")

# --- Write player match stats ---
player_table = "lake.analytics.player_match_stats"
df_player_stats.writeTo(player_table).createOrReplace()
print(f"Successfully wrote {df_player_stats.count()} rows to Iceberg table {player_table}")

### 6. Verify via SQL

In [ ]:
# Query the enriched match statistics table
print("=== Match Statistics (top 10 by total goals) ===")
spark.sql(f"""
    SELECT game_id, date, home_team, away_team, 
           home_score, away_score, total_goals, winner,
           venue, home_possession_pct, away_possession_pct,
           home_shots_on_target, away_shots_on_target
    FROM {match_table}
    ORDER BY total_goals DESC
    LIMIT 10
""").show(truncate=False)

# Query top goal scorers from player stats
print("\n=== Top Scorers (season total) ===")
spark.sql(f"""
    SELECT player, team,
           SUM(total_goals) as season_goals,
           SUM(goal_assists) as season_assists,
           COUNT(*) as appearances
    FROM {player_table}
    GROUP BY player, team
    HAVING SUM(total_goals) > 0
    ORDER BY season_goals DESC
    LIMIT 15
""").show(truncate=False)

# Show total column count for reference
ms_cols = spark.sql(f"SELECT * FROM {match_table} LIMIT 1").columns
ps_cols = spark.sql(f"SELECT * FROM {player_table} LIMIT 1").columns
print(f"\nmatch_statistics: {len(ms_cols)} columns")
print(f"player_match_stats: {len(ps_cols)} columns")